# Member 2 – Forecast + GPU Risk Model v2

## Objective

Generate dengue inspection priorities using:

- Google BigQuery ML (ARIMA_PLUS Forecast)
- NVIDIA T4 GPU
- RAPIDS cuDF
- Google BigQuery

### Input Tables

- dengue_ew.features
- dengue_ew.forecast_14d

### Output Table

- dengue_ew.inspection_priority_v2

### Workflow

Features
↓
Forecast (ARIMA_PLUS)
↓
GPU Risk Scoring
↓
Inspection Priority
↓
BigQuery



In [1]:
# Install GPU libraries

!pip -q install cudf-cu12 dask-cudf-cu12 cupy-cuda12x google-cloud-bigquery

In [2]:
import cudf
import cupy as cp
import pandas as pd
import numpy as np

print("cuDF Version:", cudf.__version__)
print("GPU Ready ✅")

cuDF Version: 26.02.01
GPU Ready ✅


In [4]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
from google.colab import auth

auth.authenticate_user()

print("Authenticated ✅")

Authenticated ✅


In [6]:
from google.cloud import bigquery

PROJECT_ID = "dengue-early-warning"

client = bigquery.Client(project=PROJECT_ID)

print("Connected to BigQuery ✅")

Connected to BigQuery ✅


In [7]:
query = """
SELECT table_name
FROM `dengue-early-warning.dengue_ew.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

tables = client.query(query).to_dataframe(create_bqstorage_client=False)

tables

,table_name
0,features
1,forecast_14d
2,gemini_recommendations
3,inspection_priority
4,inspection_priority_v2
5,inspection_with_ai
6,telemetry_daily


In [9]:
query = """
SELECT COUNT(*) AS total_rows
FROM `dengue-early-warning.dengue_ew.forecast_14d`
"""

client.query(query).to_dataframe(create_bqstorage_client=False)

,total_rows
0,36624


In [10]:
query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.forecast_14d`
LIMIT 10
"""

forecast_df = client.query(query).to_dataframe(create_bqstorage_client=False)

forecast_df.head()

,cell_id,forecast_timestamp,forecast_value,standard_error,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,confidence_interval_lower_bound,confidence_interval_upper_bound
0,1.2645_103.6125,2020-11-07 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0
1,1.2645_103.6125,2020-11-08 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0
2,1.2645_103.6125,2020-11-09 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0
3,1.2645_103.6125,2020-11-10 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0
4,1.2645_103.6125,2020-11-11 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0


In [13]:
query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.features`
LIMIT 1000
"""

features_df = client.query(query).to_dataframe(create_bqstorage_client=False)

print(features_df.shape)
features_df.head()

(1000, 9)


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence
0,1.2645_103.6125,1.2645,103.6125,2015-07-09,0.0,10.428061,0.0,NaN,0.0
1,1.2645_103.6125,1.2645,103.6125,2015-07-10,0.0,9.605557,0.0,NaN,0.0
2,1.2645_103.6125,1.2645,103.6125,2015-07-11,0.0,2.791308,0.0,NaN,0.0
3,1.2645_103.6125,1.2645,103.6125,2015-07-12,0.0,1.475074,0.0,NaN,0.0
4,1.2645_103.6125,1.2645,103.6125,2015-07-13,0.0,3.171638,0.0,NaN,0.0


In [14]:
# Latest feature row per cell

features_latest = (
    features_df
    .sort_values("date")
    .groupby("cell_id", as_index=False)
    .last()
)

print("Latest Feature Rows:", len(features_latest))
features_latest.head()


Latest Feature Rows: 1


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence
0,1.2645_103.6125,1.2645,103.6125,2018-04-03,0.0,2.501075,0.0,13.275753,0.0


In [15]:
# Latest forecast per cell

forecast_latest = (
    forecast_df
    .sort_values("forecast_timestamp")
    .groupby("cell_id", as_index=False)
    .last()
)

print("Latest Forecast Rows:", len(forecast_latest))
forecast_latest.head()


Latest Forecast Rows: 1


,cell_id,forecast_timestamp,forecast_value,standard_error,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,confidence_interval_lower_bound,confidence_interval_upper_bound
0,1.2645_103.6125,2020-11-16 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0


In [16]:
merged_df = features_latest.merge(
    forecast_latest[["cell_id", "forecast_value"]],
    on="cell_id",
    how="left"
)

merged_df["forecast_value"] = merged_df["forecast_value"].fillna(0)

print("Merged Rows:", len(merged_df))
merged_df.head()

Merged Rows: 1


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,forecast_value
0,1.2645_103.6125,1.2645,103.6125,2018-04-03,0.0,2.501075,0.0,13.275753,0.0,0.0


In [17]:
query = """
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY cell_id
               ORDER BY date DESC
           ) AS rn
    FROM `dengue-early-warning.dengue_ew.features`
)
WHERE rn = 1
"""

features_latest = client.query(query).to_dataframe(create_bqstorage_client=False)

print(features_latest.shape)
features_latest.head()

(2616, 10)


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,rn
0,1.2645_103.6125,1.26450,103.61250,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1
1,1.26675_103.815,1.26675,103.81500,2020-11-06,0.0,7.001018,0.0,15.896079,0.001024,1
2,1.26675_103.824,1.26675,103.82400,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1
3,1.269_103.61925,1.26900,103.61925,2020-11-06,0.0,7.001018,0.0,15.896079,0.001535,1
4,1.269_103.82175,1.26900,103.82175,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1


In [18]:
query = """
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY cell_id
               ORDER BY forecast_timestamp DESC
           ) AS rn
    FROM `dengue-early-warning.dengue_ew.forecast_14d`
)
WHERE rn = 1
"""

forecast_latest = client.query(query).to_dataframe(create_bqstorage_client=False)

print(forecast_latest.shape)
forecast_latest.head()

(2616, 10)


,cell_id,forecast_timestamp,forecast_value,standard_error,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,confidence_interval_lower_bound,confidence_interval_upper_bound,rn
0,1.2645_103.6125,2020-11-20 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0,1
1,1.26675_103.815,2020-11-20 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0,1
2,1.26675_103.824,2020-11-20 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0,1
3,1.269_103.61925,2020-11-20 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0,1
4,1.269_103.82175,2020-11-20 00:00:00+00:00,0.0,0.0,0.9,0.0,0.0,0.0,0.0,1


In [19]:
merged_df = features_latest.merge(
    forecast_latest[["cell_id", "forecast_value"]],
    on="cell_id",
    how="left"
)

merged_df["forecast_value"] = merged_df["forecast_value"].fillna(0)

print("Merged Shape:", merged_df.shape)
merged_df.head()


Merged Shape: (2616, 11)


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,rn,forecast_value
0,1.2645_103.6125,1.26450,103.61250,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1,0.0
1,1.26675_103.815,1.26675,103.81500,2020-11-06,0.0,7.001018,0.0,15.896079,0.001024,1,0.0
2,1.26675_103.824,1.26675,103.82400,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1,0.0
3,1.269_103.61925,1.26900,103.61925,2020-11-06,0.0,7.001018,0.0,15.896079,0.001535,1,0.0
4,1.269_103.82175,1.26900,103.82175,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1,0.0


In [21]:
gdf = cudf.from_pandas(merged_df)

print("GPU Rows:", len(gdf))
gdf.head()

GPU Rows: 2616


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,rn,forecast_value
0,1.2645_103.6125,1.26450,103.61250,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1,0.0
1,1.26675_103.815,1.26675,103.81500,2020-11-06,0.0,7.001018,0.0,15.896079,0.001024,1,0.0
2,1.26675_103.824,1.26675,103.82400,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1,0.0
3,1.269_103.61925,1.26900,103.61925,2020-11-06,0.0,7.001018,0.0,15.896079,0.001535,1,0.0
4,1.269_103.82175,1.26900,103.82175,2020-11-06,0.0,7.001018,0.0,15.896079,0.002047,1,0.0


In [22]:
cols = [
    "case_density_14d",
    "rain_lag_7to14d",
    "recurrence",
    "forecast_value"
]

for c in cols:
    gdf[c] = gdf[c].fillna(0)

gdf[cols].isna().sum()

case_density_14d    0
rain_lag_7to14d     0
recurrence          0
forecast_value      0
dtype: int64

In [23]:
gdf["risk_score"] = (
      gdf["case_density_14d"] * 0.40
    + gdf["rain_lag_7to14d"] * 0.20
    + gdf["recurrence"] * 0.15
    + gdf["forecast_value"] * 0.25
)

gdf["risk_score"] = gdf["risk_score"].round(2)

gdf[[
    "case_density_14d",
    "rain_lag_7to14d",
    "recurrence",
    "forecast_value",
    "risk_score"
]].head()

,case_density_14d,rain_lag_7to14d,recurrence,forecast_value,risk_score
0,0.0,15.896079,0.002047,0.0,3.18
1,0.0,15.896079,0.001024,0.0,3.18
2,0.0,15.896079,0.002047,0.0,3.18
3,0.0,15.896079,0.001535,0.0,3.18
4,0.0,15.896079,0.002047,0.0,3.18


In [24]:
risk = (
    gdf
    .sort_values("risk_score", ascending=False)
    .reset_index(drop=True)
)

risk["rank"] = risk.index + 1

risk.head(10)

,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,rn,forecast_value,risk_score,rank
0,1.33425_103.8825,1.33425,103.88250,2020-11-06,86.0,7.001018,86.0,15.896079,0.100819,1,4.081365,38.61,1
1,1.34775_103.95225,1.34775,103.95225,2020-11-06,76.0,7.001018,76.0,15.896079,0.063460,1,3.601444,34.49,2
2,1.314_103.9275,1.31400,103.92750,2020-11-06,51.0,7.001018,51.0,15.896079,0.075230,1,2.419172,24.20,3
3,1.314_103.85325,1.31400,103.85325,2020-11-06,48.0,7.001018,48.0,15.896079,0.315763,1,2.275486,23.00,4
4,1.3815_103.9365,1.38150,103.93650,2020-11-06,45.0,7.001018,45.0,15.896079,0.026612,1,2.133953,21.72,5
5,1.395_103.869,1.39500,103.86900,2020-11-06,44.0,7.001018,44.0,15.896079,0.035312,1,2.088660,21.31,6
6,1.27575_103.8285,1.27575,103.82850,2020-11-06,41.0,7.001018,41.0,15.896079,0.031218,1,1.944138,20.07,7
7,1.37925_103.9365,1.37925,103.93650,2020-11-06,40.0,7.001018,40.0,15.896079,0.025077,1,1.897390,19.66,8
8,1.34775_103.95,1.34775,103.95000,2020-11-06,39.0,7.001018,39.0,15.896079,0.048618,1,1.695488,19.21,9
9,1.368_103.93425,1.36800,103.93425,2020-11-06,37.0,7.001018,37.0,15.896079,0.018936,1,1.753382,18.42,10


In [25]:
risk["risk_level"] = "Low"

risk.loc[risk["risk_score"] >= 10, "risk_level"] = "Medium"
risk.loc[risk["risk_score"] >= 20, "risk_level"] = "High"
risk.loc[risk["risk_score"] >= 30, "risk_level"] = "Critical"

risk.head(10)

,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,rn,forecast_value,risk_score,rank,risk_level
0,1.33425_103.8825,1.33425,103.88250,2020-11-06,86.0,7.001018,86.0,15.896079,0.100819,1,4.081365,38.61,1,Critical
1,1.34775_103.95225,1.34775,103.95225,2020-11-06,76.0,7.001018,76.0,15.896079,0.063460,1,3.601444,34.49,2,Critical
2,1.314_103.9275,1.31400,103.92750,2020-11-06,51.0,7.001018,51.0,15.896079,0.075230,1,2.419172,24.20,3,High
3,1.314_103.85325,1.31400,103.85325,2020-11-06,48.0,7.001018,48.0,15.896079,0.315763,1,2.275486,23.00,4,High
4,1.3815_103.9365,1.38150,103.93650,2020-11-06,45.0,7.001018,45.0,15.896079,0.026612,1,2.133953,21.72,5,High
5,1.395_103.869,1.39500,103.86900,2020-11-06,44.0,7.001018,44.0,15.896079,0.035312,1,2.088660,21.31,6,High
6,1.27575_103.8285,1.27575,103.82850,2020-11-06,41.0,7.001018,41.0,15.896079,0.031218,1,1.944138,20.07,7,High
7,1.37925_103.9365,1.37925,103.93650,2020-11-06,40.0,7.001018,40.0,15.896079,0.025077,1,1.897390,19.66,8,Medium
8,1.34775_103.95,1.34775,103.95000,2020-11-06,39.0,7.001018,39.0,15.896079,0.048618,1,1.695488,19.21,9,Medium
9,1.368_103.93425,1.36800,103.93425,2020-11-06,37.0,7.001018,37.0,15.896079,0.018936,1,1.753382,18.42,10,Medium


In [26]:
risk_pd = risk.to_pandas()

risk_pd.head()


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,rn,forecast_value,risk_score,rank,risk_level
0,1.33425_103.8825,1.33425,103.88250,2020-11-06,86.0,7.001018,86.0,15.896079,0.100819,1,4.081365,38.61,1,Critical
1,1.34775_103.95225,1.34775,103.95225,2020-11-06,76.0,7.001018,76.0,15.896079,0.063460,1,3.601444,34.49,2,Critical
2,1.314_103.9275,1.31400,103.92750,2020-11-06,51.0,7.001018,51.0,15.896079,0.075230,1,2.419172,24.20,3,High
3,1.314_103.85325,1.31400,103.85325,2020-11-06,48.0,7.001018,48.0,15.896079,0.315763,1,2.275486,23.00,4,High
4,1.3815_103.9365,1.38150,103.93650,2020-11-06,45.0,7.001018,45.0,15.896079,0.026612,1,2.133953,21.72,5,High


In [27]:
from google.cloud import bigquery

table_id = "dengue-early-warning.dengue_ew.inspection_priority_v2"

job = client.load_table_from_dataframe(risk_pd, table_id)

job.result()

print("✅ Uploaded Successfully")

BadRequest: 400 POST https://bigquery.googleapis.com/upload/bigquery/v2/projects/dengue-early-warning/jobs?uploadType=multipart: Provided Schema does not match Table dengue-early-warning:dengue_ew.inspection_priority_v2. Cannot add fields (field: rn)

In [28]:
risk_pd = risk.to_pandas()

risk_pd = risk_pd.drop(columns=["rn"], errors="ignore")

risk_pd.head()


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,forecast_value,risk_score,rank,risk_level
0,1.33425_103.8825,1.33425,103.88250,2020-11-06,86.0,7.001018,86.0,15.896079,0.100819,4.081365,38.61,1,Critical
1,1.34775_103.95225,1.34775,103.95225,2020-11-06,76.0,7.001018,76.0,15.896079,0.063460,3.601444,34.49,2,Critical
2,1.314_103.9275,1.31400,103.92750,2020-11-06,51.0,7.001018,51.0,15.896079,0.075230,2.419172,24.20,3,High
3,1.314_103.85325,1.31400,103.85325,2020-11-06,48.0,7.001018,48.0,15.896079,0.315763,2.275486,23.00,4,High
4,1.3815_103.9365,1.38150,103.93650,2020-11-06,45.0,7.001018,45.0,15.896079,0.026612,2.133953,21.72,5,High


In [30]:
risk_pd.columns

Index(['cell_id', 'lat', 'lon', 'date', 'case_count', 'rain_mm',
       'case_density_14d', 'rain_lag_7to14d', 'recurrence', 'forecast_value',
       'risk_score', 'rank', 'risk_level'],
      dtype='object')

In [32]:
risk_pd = risk_pd.drop(columns=["rn"], errors="ignore")


In [33]:
risk_pd.columns

Index(['cell_id', 'lat', 'lon', 'date', 'case_count', 'rain_mm',
       'case_density_14d', 'rain_lag_7to14d', 'recurrence', 'forecast_value',
       'risk_score', 'rank', 'risk_level'],
      dtype='object')

In [34]:
from google.cloud import bigquery

table_id = "dengue-early-warning.dengue_ew.inspection_priority_v2"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    risk_pd,
    table_id,
    job_config=job_config
)

job.result()

print("✅ Uploaded Successfully")

✅ Uploaded Successfully


# ✅ Member 2 Completed - AI Risk Forecast & Inspection Prioritization

## Objective
Develop a GPU-accelerated AI pipeline to prioritize dengue inspection areas by combining historical outbreak features with BigQuery ML (ARIMA_PLUS) forecasts.

---

## Workflow

1. Connected to Google BigQuery
2. Loaded engineered feature dataset
3. Loaded 14-day ARIMA_PLUS forecast
4. Selected latest forecast for each grid cell
5. Merged forecast with historical features
6. Calculated AI Risk Score
7. Ranked all grid cells by inspection priority
8. Assigned Risk Levels (Low, Medium, High, Critical)
9. Uploaded final results back to BigQuery

---

## Technologies Used

- Google Colab (T4 GPU)
- RAPIDS cuDF
- CuPy
- Pandas
- BigQuery
- BigQuery ML (ARIMA_PLUS)

---

## Input Tables

- dengue_ew.features
- dengue_ew.forecast_14d

---

## Output Table

- dengue_ew.inspection_priority_v2

---

## Output Fields

- cell_id
- latitude
- longitude
- date
- case_count
- rainfall
- recurrence
- forecast_value
- risk_score
- rank
- risk_level

---

## Status

**Member 2 Implementation Completed Successfully**

✔ GPU Risk Scoring  
✔ Forecast Integration  
✔ Inspection Priority Ranking  
✔ BigQuery Upload Completed